In [1]:
import pandas as pd
from prophet import Prophet
import joblib

c:\Users\munashe\Projects\eharvest-main\eharvest_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
# 1. Load the dataset
df = pd.read_csv('wfp_food_prices_zwe.csv')

In [3]:
df.head

<bound method NDFrame.head of              date               admin1        admin2      market  market_id  \
0      2010-01-15                  NaN           NaN      Kombai        723   
1      2010-01-15                  NaN           NaN     Mandava        722   
2      2010-01-15                  NaN           NaN     Mucheke        719   
3      2010-01-15                  NaN           NaN  Murombedzi        710   
4      2010-01-15                  NaN           NaN    Tshovani        721   
...           ...                  ...           ...         ...        ...   
18136  2025-12-15  Mashonaland Central  Mount Darwin     Karanda       5733   
18137  2025-12-15  Mashonaland Central  Mount Darwin     Karanda       5733   
18138  2025-12-15  Mashonaland Central  Mount Darwin     Karanda       5733   
18139  2025-12-15  Mashonaland Central  Mount Darwin     Karanda       5733   
18140  2025-12-15  Mashonaland Central  Mount Darwin     Karanda       5733   

       latitude  long

In [4]:
# 2. Pre-processing
# Prophet requires 'ds' (date) and 'y' (target)
df['ds'] = pd.to_datetime(df['date'])
df['y'] = df['usdprice']
df['admin1'] = df['admin1'].fillna('Unknown')

# We need to aggregate to avoid duplicate dates for the same commodity/region
df_agg = df.groupby(['ds', 'commodity', 'admin1'])['y'].mean().reset_index()

In [5]:
# 3. Handle Categorical Data (One-Hot Encoding)
# This allows one model to understand many commodities and regions
df_prophet = pd.get_dummies(df_agg, columns=['commodity', 'admin1'])
commodity_cols = [col for col in df_prophet.columns if col.startswith('commodity_') or col.startswith('admin1_')]

In [6]:
# 4. Initialize and Train the Global Model
model = Prophet(yearly_seasonality=True, daily_seasonality=False)

# Add each commodity as a 'regressor'
for col in commodity_cols:
    model.add_regressor(col)

model.fit(df_prophet)

23:04:03 - cmdstanpy - INFO - Chain [1] start processing
23:04:09 - cmdstanpy - INFO - Chain [1] done processing


In [7]:
# 5. Save the Model and the Column List
# We save the columns so the API knows the exact order of the "switches"
joblib.dump(model, 'demand_forecast_model.pkl')
joblib.dump(commodity_cols, 'forecast_features.pkl')

print("Model and features saved successfully!")

Model and features saved successfully!
